# ShopSense — Final Recommendation Engine (CF + SASRec)

Consolidated final notebook. Everything below has been fixed and verified
against three real bugs found during development -- documented inline at
the point each one is fixed, not swept under the rug.

**Bugs fixed in this version:**
1. **Leakage** in the original train/test split (per-user cutoff + assertion).
2. **`float32`/`float64` mismatch** crashing `ItemItemRecommender.fit()`.
3. **Session ordering bug**: grouping on the string `session_id` sorted
   `"_10"` before `"_9"`, silently picking the wrong "last session" for
   users with 10+ sessions. Fixed by grouping on numeric `session_num`.
4. **`filter_already_liked_items=True` evaluation bug**: this flag does
   NOT remove already-seen items -- it zeroes their score and leaves them
   as filler once real candidates run out. Since ~98% of held-out targets
   are items the user already interacted with (browse-then-buy), this
   padding artifact was inflating Recall@50/@100 with a fake cliff.
5. **SASRec NaN**: a fully-padded position has zero valid attention
   targets under the causal mask, producing NaN. `0 * NaN = NaN`, so it
   leaks into real positions across layers. Fixed by cleaning NaN between
   each transformer layer, not once after the whole model has run.

**Evaluation objective, stated explicitly:** the train/test split holds out
each user's LAST transaction; train = everything strictly before it. This
is a **next-interaction prediction task** ("what does this user do next,"
including returning to something they already looked at) -- not a
"discover something brand new" task. That distinction decides `exclude_seen`
below, and both evaluations are included, correctly scoped.


## Step 1 — Imports and data load

In [1]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize
from implicit.als import AlternatingLeastSquares
from implicit.nearest_neighbours import ItemItemRecommender
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', lambda x: f'{x:.6f}')


c:\Users\Ayush\Machine Learning projects\shope_sense\shope_sense\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
events_df = pd.read_csv('../datasets/events.csv')
events_df['timestamp'] = pd.to_datetime(events_df['timestamp'], unit='ms')
events_df = events_df.sort_values('timestamp').reset_index(drop=True)

print("Total events:", len(events_df))
events_df.head()


Total events: 2756101


,timestamp,visitorid,event,itemid,transactionid
0,2015-05-03 03:00:04.384,693516,addtocart,297662,NaN
1,2015-05-03 03:00:11.289,829044,view,60987,NaN
2,2015-05-03 03:00:13.048,652699,view,252860,NaN
3,2015-05-03 03:00:24.154,1125936,view,33661,NaN
4,2015-05-03 03:00:26.228,693516,view,297662,NaN


## Step 2 — Data audit

In [3]:
print("=" * 60)
print("EVENT COUNTS")
print("=" * 60)
print(events_df['event'].value_counts())

n_users_raw = events_df['visitorid'].nunique()
n_items_raw = events_df['itemid'].nunique()
print(f"\nUnique visitors: {n_users_raw:,}")
print(f"Unique items:    {n_items_raw:,}")

interactions_per_user = events_df.groupby('visitorid').size()
print("\nCold-start breakdown:")
for t in [1, 2, 3, 5, 10]:
    n = (interactions_per_user >= t).sum()
    print(f"  Users with >= {t} events: {n:,} ({n / n_users_raw * 100:.1f}%)")

n_pairs = events_df.drop_duplicates(['visitorid', 'itemid']).shape[0]
sparsity_raw = 1 - n_pairs / (n_users_raw * n_items_raw)
print(f"\nRaw user-item sparsity: {sparsity_raw:.6%}")


EVENT COUNTS
event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64

Unique visitors: 1,407,580
Unique items:    235,061

Cold-start breakdown:
  Users with >= 1 events: 1,407,580 (100.0%)
  Users with >= 2 events: 406,020 (28.8%)
  Users with >= 3 events: 200,028 (14.2%)
  Users with >= 5 events: 81,620 (5.8%)
  Users with >= 10 events: 23,241 (1.7%)

Raw user-item sparsity: 99.999352%


## Step 3 — Leakage-free chronological split

**Bug fixed here:** the original split excluded only the exact
`(visitorid, itemid, timestamp)` row of the held-out transaction. Every
other event for that user -- including ones AFTER the cutoff -- stayed in
training. Fixed with a per-user cutoff timestamp and an assertion that
fails immediately if this is ever broken again.


In [4]:
transactions = events_df[events_df['event'] == 'transaction'].copy()
test_df = (
    transactions.sort_values('timestamp')
    .groupby('visitorid')
    .tail(1)
    .reset_index(drop=True)
)
print(f"Test users (held-out last transaction): {len(test_df):,}")

cutoff_by_user = test_df.set_index('visitorid')['timestamp']
events_df['cutoff'] = events_df['visitorid'].map(cutoff_by_user)

train_mask = events_df['cutoff'].isna() | (events_df['timestamp'] < events_df['cutoff'])
train_events = events_df[train_mask].drop(columns='cutoff').copy()

print(f"Train events: {len(train_events):,} / {len(events_df):,} total events")


Test users (held-out last transaction): 11,719
Train events: 2,707,377 / 2,756,101 total events


In [5]:
check = train_events.merge(
    cutoff_by_user.rename('cutoff'), left_on='visitorid', right_index=True, how='inner'
)
assert (check['timestamp'] < check['cutoff']).all(), \
    "LEAKAGE: some training events occur at or after the held-out transaction timestamp."
print("PASS: every training event for every test user occurs strictly before their held-out transaction.")


PASS: every training event for every test user occurs strictly before their held-out transaction.


In [6]:
# How much of the test set is a repeat-interaction target vs. a genuinely
# novel item? This defines the two evaluation scopes used later.
already_seen_mask = []
train_by_user = train_events.groupby('visitorid')['itemid'].apply(set)

for _, row in test_df.iterrows():
    uid, target = row['visitorid'], row['itemid']
    seen = train_by_user.get(uid, set())
    already_seen_mask.append(target in seen)

test_df['target_already_seen'] = already_seen_mask
n_repeat = test_df['target_already_seen'].sum()
n_novel = (~test_df['target_already_seen']).sum()
print(f"Test targets that are REPEAT interactions (already seen pre-cutoff): {n_repeat:,} ({n_repeat/len(test_df)*100:.1f}%)")
print(f"Test targets that are NOVEL (never seen pre-cutoff):               {n_novel:,} ({n_novel/len(test_df)*100:.1f}%)")
print()
print("This is why exclude_seen=True and exclude_seen=False measure DIFFERENT objectives here,")
print("not just 'stricter vs looser' -- with this few novel targets, exclude_seen=True")
print("evaluated on the FULL test set is structurally unwinnable for ~98% of users.")


Test targets that are REPEAT interactions (already seen pre-cutoff): 11,476 (97.9%)
Test targets that are NOVEL (never seen pre-cutoff):               243 (2.1%)

This is why exclude_seen=True and exclude_seen=False measure DIFFERENT objectives here,
not just 'stricter vs looser' -- with this few novel targets, exclude_seen=True
evaluated on the FULL test set is structurally unwinnable for ~98% of users.


## Step 4 — Weight events and build interactions

In [7]:
EVENT_WEIGHTS = {'view': 1, 'addtocart': 3, 'transaction': 5}
train_events['weight'] = train_events['event'].map(EVENT_WEIGHTS)

interaction_df = (
    train_events.groupby(['visitorid', 'itemid'], as_index=False)['weight'].sum()
)
print(f"Train interaction rows (unique user-item pairs): {len(interaction_df):,}")


Train interaction rows (unique user-item pairs): 2,127,517


## Step 5 — ID mappings and sparse matrices

**Bug fixed here:** `implicit`'s `ItemItemRecommender` requires `float64`
("double") and raises a buffer dtype mismatch on `float32`. Standardized on
`float64` from the start so every downstream consumer sees the same,
correct dtype.


In [8]:
user_ids = interaction_df['visitorid'].unique()
item_ids = interaction_df['itemid'].unique()

user_to_idx = {u: i for i, u in enumerate(user_ids)}
item_to_idx = {it: i for i, it in enumerate(item_ids)}
idx_to_user = {i: u for u, i in user_to_idx.items()}
idx_to_item = {i: it for it, i in item_to_idx.items()}

interaction_df['user_idx'] = interaction_df['visitorid'].map(user_to_idx)
interaction_df['item_idx'] = interaction_df['itemid'].map(item_to_idx)

n_users = len(user_to_idx)
n_items = len(item_to_idx)

user_item_matrix = csr_matrix(
    (
        interaction_df['weight'].astype(np.float64),
        (interaction_df['user_idx'], interaction_df['item_idx']),
    ),
    shape=(n_users, n_items),
)
item_user_matrix = user_item_matrix.T.tocsr()

print(f"Users: {n_users:,}   Items: {n_items:,}")
print(f"user_item_matrix: {user_item_matrix.shape}, dtype={user_item_matrix.dtype}")


Users: 1,407,477   Items: 234,561
user_item_matrix: (1407477, 234561), dtype=float64


In [ ]:

assert n_users == len(user_to_idx)
assert n_items == len(item_to_idx)
assert len(item_to_idx) == len(idx_to_item)
assert user_item_matrix.shape == (n_users, n_items)
assert item_user_matrix.shape == (n_items, n_users)
assert user_item_matrix.dtype == np.float64, \
    "user_item_matrix must be float64 -- implicit's ItemItemRecommender will raise a buffer dtype mismatch on float32."
print("PASS: all matrix and mapping invariants hold.")

PASS: all matrix and mapping invariants hold.


## Step 6 — Shared evaluation harness

In [10]:
K_VALUES = [5, 10, 20, 50, 100]

eval_user_idx, eval_targets, eval_already_seen = [], [], []
for _, row in test_df.iterrows():
    uid = row['visitorid']
    if uid in user_to_idx:
        eval_user_idx.append(user_to_idx[uid])
        eval_targets.append(row['itemid'])
        eval_already_seen.append(row['target_already_seen'])

eval_already_seen = np.array(eval_already_seen)
print(f"Evaluable test users: {len(eval_user_idx):,} / {len(test_df):,}")
print(f"  of which, novel-target subset: {(~eval_already_seen).sum():,}")

def evaluate(recommend_fn, user_subset=None, k_values=K_VALUES):
    """recommend_fn(user_idx, N) -> list of internal item indices, ranked.
    user_subset: optional boolean mask over eval_user_idx/eval_targets, to
    restrict evaluation to e.g. only the novel-target users."""
    max_k = max(k_values)
    hits = {k: 0 for k in k_values}
    total = 0
    indices = range(len(eval_user_idx)) if user_subset is None else np.where(user_subset)[0]
    for i in indices:
        user_idx, actual_item = eval_user_idx[i], eval_targets[i]
        ranked_idx = recommend_fn(user_idx, max_k)
        ranked_items = [idx_to_item[int(j)] for j in ranked_idx]
        for k in k_values:
            if actual_item in ranked_items[:k]:
                hits[k] += 1
        total += 1
    return {f"Recall@{k}": hits[k] / total for k in k_values}, total


Evaluable test users: 11,616 / 11,719
  of which, novel-target subset: 140


## Step 7 — Baselines

In [11]:
popularity_rank = train_events.groupby('itemid').size().sort_values(ascending=False)
popular_item_ids = popularity_rank.index.tolist()

txn_popularity_rank = (
    train_events[train_events['event'] == 'transaction']
    .groupby('itemid').size().sort_values(ascending=False)
)
txn_popular_item_ids = txn_popularity_rank.index.tolist()

def popularity_recommend(user_idx, N):
    return [item_to_idx[i] for i in popular_item_ids[:N] if i in item_to_idx]

def txn_popularity_recommend(user_idx, N):
    return [item_to_idx[i] for i in txn_popular_item_ids[:N] if i in item_to_idx]

pop_results, _ = evaluate(popularity_recommend)
txn_pop_results, _ = evaluate(txn_popularity_recommend)
print("Popularity:            ", pop_results)
print("Transaction popularity:", txn_pop_results)


Popularity:             {'Recall@5': 0.007145316804407714, 'Recall@10': 0.010330578512396695, 'Recall@20': 0.01997245179063361, 'Recall@50': 0.03710399449035812, 'Recall@100': 0.058798209366391185}
Transaction popularity: {'Recall@5': 0.0109331955922865, 'Recall@10': 0.017389807162534434, 'Recall@20': 0.02952823691460055, 'Recall@50': 0.04803719008264463, 'Recall@100': 0.06585743801652892}


## Step 8 — User-user CF (comparison only, not used in production)

In [12]:
MIN_INTERACTIONS_UU = 5
counts_all = interaction_df.groupby('visitorid').size()
uu_eligible_users = set(counts_all[counts_all >= MIN_INTERACTIONS_UU].index)
print(f"User-user CF eligible pool: {len(uu_eligible_users):,} / {n_users:,} users")

uu_df = interaction_df[interaction_df['visitorid'].isin(uu_eligible_users)].copy()
uu_user_ids = uu_df['visitorid'].unique()
uu_user_to_idx = {u: i for i, u in enumerate(uu_user_ids)}
uu_df['uu_idx'] = uu_df['visitorid'].map(uu_user_to_idx)

uu_matrix = csr_matrix(
    (uu_df['weight'].astype(np.float32), (uu_df['uu_idx'], uu_df['item_idx'])),
    shape=(len(uu_user_ids), n_items),
)
uu_matrix_norm = normalize(uu_matrix, axis=1)

def user_user_recommend(user_idx, N, top_k_neighbors=20):
    visitor_id = idx_to_user[user_idx]
    if visitor_id not in uu_user_to_idx:
        return []
    uu_idx = uu_user_to_idx[visitor_id]
    sims = (uu_matrix_norm @ uu_matrix_norm[uu_idx].T).toarray().ravel()
    sims[uu_idx] = -1
    top_neighbors = np.argpartition(sims, -top_k_neighbors)[-top_k_neighbors:]
    top_neighbors = top_neighbors[np.argsort(-sims[top_neighbors])]
    seen = set(user_item_matrix[user_idx].indices)
    scores = {}
    for n_idx in top_neighbors:
        sim = sims[n_idx]
        if sim <= 0:
            continue
        row = uu_matrix[n_idx]
        for item_idx, w in zip(row.indices, row.data):
            if item_idx in seen:
                continue
            scores[item_idx] = scores.get(item_idx, 0) + sim * w
    ranked = sorted(scores.items(), key=lambda x: -x[1])[:N]
    return [i for i, _ in ranked]

uu_results, _ = evaluate(user_user_recommend)
print("User-user CF:", uu_results)


User-user CF eligible pool: 38,721 / 1,407,477 users
User-user CF: {'Recall@5': 0.00043044077134986227, 'Recall@10': 0.00043044077134986227, 'Recall@20': 0.0007747933884297521, 'Recall@50': 0.001119146005509642, 'Recall@100': 0.0012913223140495868}


## Step 9 — Item-item CF (production candidate generator)

**`exclude_seen=False` is the default here.** The task is next-interaction
prediction, and ~98% of held-out targets are items the user already
interacted with -- a real, dominant behavior pattern, not noise to filter
out. `filter_already_liked_items=True` also does NOT actually remove
already-seen items in `implicit` -- it zeroes their score and leaves them
as filler once real candidates run out, which produced the fake
Recall@20-to-@50 cliff seen earlier in this project. Evaluating with
`exclude_seen=False` avoids that broken padding path entirely, using
genuinely-scored candidates instead.


In [13]:
item_model = ItemItemRecommender(K=50, num_threads=4)
item_model.fit(user_item_matrix)
assert item_model.similarity.shape == (n_items, n_items)
print("PASS: item-item similarity matrix shape:", item_model.similarity.shape)

def item_item_recommend(user_idx, N, exclude_seen=False):
    ranked_idx, scores = item_model.recommend(
        userid=user_idx,
        user_items=user_item_matrix[user_idx],
        N=N,
        filter_already_liked_items=exclude_seen,
    )
    return list(ranked_idx)

item_item_results, _ = evaluate(lambda u, N: item_item_recommend(u, N, exclude_seen=False))
print("Item-item CF (exclude_seen=False, next-interaction objective):", item_item_results)


100%|██████████| 234561/234561 [00:01<00:00, 166729.08it/s]


PASS: item-item similarity matrix shape: (234561, 234561)
Item-item CF (exclude_seen=False, next-interaction objective): {'Recall@5': 0.849862258953168, 'Recall@10': 0.8904097796143251, 'Recall@20': 0.9177858126721763, 'Recall@50': 0.9413739669421488, 'Recall@100': 0.9745179063360881}


In [14]:
# Secondary, correctly-scoped evaluation: exclude_seen=True, but ONLY
# against the subset of test users whose target was genuinely novel.
# Evaluating True against the FULL test set is unfair by construction --
# ~98% of those cases are unwinnable regardless of model quality, since the
# true answer is excluded before the model ever sees it.
novel_subset = ~eval_already_seen
item_item_discovery_results, n_novel_eval = evaluate(
    lambda u, N: item_item_recommend(u, N, exclude_seen=True),
    user_subset=novel_subset,
)
print(f"Item-item CF (exclude_seen=True, discovery objective, n={n_novel_eval} novel-target users):")
print(item_item_discovery_results)


Item-item CF (exclude_seen=True, discovery objective, n=140 novel-target users):
{'Recall@5': 0.14285714285714285, 'Recall@10': 0.15, 'Recall@20': 0.22142857142857142, 'Recall@50': 0.24285714285714285, 'Recall@100': 0.2785714285714286}


## Step 10 — ALS

Same `exclude_seen=False` reasoning applies here. Orientation is verified
by fitting both matrix orientations and keeping whichever produces factor
shapes matching the known user/item counts, rather than guessing.


In [15]:
def fit_als_correctly(user_item_matrix, item_user_matrix, n_users, n_items, **kwargs):
    for name, matrix in [
        ("item_user_matrix (items x users)", item_user_matrix),
        ("user_item_matrix (users x items)", user_item_matrix),
    ]:
        model = AlternatingLeastSquares(**kwargs)
        model.fit(matrix)
        if model.user_factors.shape[0] == n_users and model.item_factors.shape[0] == n_items:
            print(f"PASS: correct ALS orientation -> fit on {name}")
            return model
    raise RuntimeError("ALS produced misaligned factor shapes under both orientations.")

als_model = fit_als_correctly(
    user_item_matrix, item_user_matrix, n_users, n_items,
    factors=64, regularization=0.05, iterations=20, random_state=42,
)
assert als_model.user_factors.shape[0] == n_users
assert als_model.item_factors.shape[0] == n_items


100%|██████████| 20/20 [01:16<00:00,  3.81s/it]

PASS: correct ALS orientation -> fit on user_item_matrix (users x items)


In [16]:
def als_recommend(user_idx, N, exclude_seen=False):
    ranked_idx, _ = als_model.recommend(
        userid=user_idx, user_items=user_item_matrix[user_idx],
        N=N, filter_already_liked_items=exclude_seen,
    )
    return list(ranked_idx)

als_results, _ = evaluate(lambda u, N: als_recommend(u, N, exclude_seen=False))
print("ALS:", als_results)


ALS: {'Recall@5': 0.2356232782369146, 'Recall@10': 0.30113636363636365, 'Recall@20': 0.3886880165289256, 'Recall@50': 0.5130853994490359, 'Recall@100': 0.6048553719008265}


## Step 11 — SASRec (session-aware re-ranker)

Sessions built with a **30-minute gap cutoff**. `session_num` is numeric,
not a string -- this matters for Step 11c below (see the comment there for
the exact bug this avoids).


In [17]:
PAD = 0
MAX_SEQ_LEN = 50
SESSION_GAP_MINUTES = 30

sessions_source = train_events.sort_values(['visitorid', 'timestamp']).copy()
sessions_source['time_diff'] = (
    sessions_source.groupby('visitorid')['timestamp'].diff().dt.total_seconds().fillna(0)
)
sessions_source['new_session'] = (sessions_source['time_diff'] > SESSION_GAP_MINUTES * 60).astype(int)
sessions_source['session_num'] = sessions_source.groupby('visitorid')['new_session'].cumsum()

sessions_source['item_idx'] = sessions_source['itemid'].map(item_to_idx)
sessions_source = sessions_source.dropna(subset=['item_idx'])
sessions_source['item_idx'] = sessions_source['item_idx'].astype(int)

session_sequences = (
    sessions_source.groupby(['visitorid', 'session_num'])['item_idx'].apply(list).tolist()
)
# Drop immediate repeats (no sequential signal) and require >= 2 items.
session_sequences = [
    [it for i, it in enumerate(seq) if i == 0 or it != seq[i - 1]]
    for seq in session_sequences
]
session_sequences = [seq[-MAX_SEQ_LEN:] for seq in session_sequences if len(seq) >= 2]

print(f"Sessions with >= 2 items: {len(session_sequences):,}")


Sessions with >= 2 items: 259,089


### Step 11a — Dataset

In [18]:
class SASRecDataset(Dataset):
    def __init__(self, sequences, n_items, max_len=MAX_SEQ_LEN):
        self.sequences = sequences
        self.n_items = n_items
        self.max_len = max_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = [i + 1 for i in self.sequences[idx]][-(self.max_len + 1):]
        input_seq, target_seq = seq[:-1], seq[1:]

        pad_len = self.max_len - len(input_seq)
        input_seq = [PAD] * pad_len + input_seq
        target_seq = [PAD] * pad_len + target_seq

        target_set = set(seq)
        neg_seq = []
        for t in target_seq:
            if t == PAD:
                neg_seq.append(PAD)
                continue
            neg = np.random.randint(1, self.n_items + 1)
            while neg in target_set:
                neg = np.random.randint(1, self.n_items + 1)
            neg_seq.append(neg)

        return (
            torch.tensor(input_seq, dtype=torch.long),
            torch.tensor(target_seq, dtype=torch.long),
            torch.tensor(neg_seq, dtype=torch.long),
        )


### Step 11b — Model

**Bug fixed here (the NaN bug):** a fully-padded position has zero valid
attention targets under the causal mask, which produces NaN for that
position. A later real position gives that padded position ~0 attention
weight -- correct -- but `0 * NaN = NaN` in floating point, not `0`, so
the corruption still leaks into the next layer's computation for real
positions. Built as separate layers (not `nn.TransformerEncoder`, which
runs all layers in one call with no chance to intervene) so NaN can be
cleaned between layers, before it has a chance to contaminate a real
position -- not once at the very end, after both layers have already run
and the damage is done.


In [19]:
class SASRec(nn.Module):
    def __init__(self, n_items, max_len=MAX_SEQ_LEN, d_model=64, n_heads=2, n_layers=2, dropout=0.2):
        super().__init__()
        self.max_len = max_len
        self.item_emb = nn.Embedding(n_items + 1, d_model, padding_idx=PAD)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
                dropout=dropout, batch_first=True,
            )
            for _ in range(n_layers)
        ])
        self.layer_norm = nn.LayerNorm(d_model)

    def forward(self, input_seq):
        positions = torch.arange(input_seq.size(1), device=input_seq.device).unsqueeze(0)
        x = self.item_emb(input_seq) + self.pos_emb(positions)
        x = self.dropout(x)

        seq_len = input_seq.size(1)
        causal_mask = torch.triu(
            torch.full((seq_len, seq_len), float('-inf'), device=input_seq.device), diagonal=1
        )
        padding_mask = (input_seq == PAD)

        for layer in self.layers:
            x = layer(x, src_mask=causal_mask, src_key_padding_mask=padding_mask)
            x = torch.nan_to_num(x, nan=0.0)  # clean between layers, not after both

        return self.layer_norm(x)

    def score_items(self, hidden, item_ids):
        item_vecs = self.item_emb(item_ids)
        return (hidden * item_vecs).sum(-1)


### Step 11c — Training and the last-session lookup

**Bug fixed here (session ordering):** grouping on the string `session_id`
(e.g. `"172_9"` vs `"172_10"`) sorts alphabetically, so `"_10"` comes before
`"_9"` -- silently returning the wrong "most recent session" for any
visitor with 10+ sessions. Grouping on the numeric `session_num` instead
sorts correctly.


In [20]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Training on:", device)

N_EPOCHS = 5
BATCH_SIZE = 128

dataset = SASRecDataset(session_sequences, n_items)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

sasrec_model = SASRec(n_items).to(device)
optimizer = torch.optim.Adam(sasrec_model.parameters(), lr=1e-3)
bce = nn.BCEWithLogitsLoss(reduction='none')

for epoch in range(N_EPOCHS):
    sasrec_model.train()
    total_loss = 0.0
    for input_seq, target_seq, neg_seq in loader:
        input_seq, target_seq, neg_seq = input_seq.to(device), target_seq.to(device), neg_seq.to(device)

        hidden = sasrec_model(input_seq)
        pos_logits = sasrec_model.score_items(hidden, target_seq)
        neg_logits = sasrec_model.score_items(hidden, neg_seq)

        mask = (target_seq != PAD).float()
        pos_loss = bce(pos_logits, torch.ones_like(pos_logits)) * mask
        neg_loss = bce(neg_logits, torch.zeros_like(neg_logits)) * mask
        loss = (pos_loss + neg_loss).sum() / mask.sum()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch + 1}/{N_EPOCHS}  loss={total_loss / len(loader):.4f}")


Training on: cpu
Epoch 1/5  loss=2.0583
Epoch 2/5  loss=1.3209
Epoch 3/5  loss=1.1018
Epoch 4/5  loss=0.9208
Epoch 5/5  loss=0.8097


In [21]:
last_session_by_user = (
    sessions_source
    .sort_values(['visitorid', 'session_num', 'timestamp'])
    .groupby(['visitorid', 'session_num'])['item_idx']
    .apply(list)
    .reset_index()
    .groupby('visitorid')
    .last()['item_idx']
    .to_dict()
)


### Step 11d — Re-ranking

**Consistency fix:** candidate generation here uses `exclude_seen=False`,
matching Step 9 -- not `True`. Passing `True` here would re-open the same
zero-score padding bug specifically for SASRec's candidate pool.


In [22]:
def sasrec_score_candidates(visitor_id, candidate_item_idx):
    session = last_session_by_user.get(visitor_id)
    if not session:
        return None

    seq = [i + 1 for i in session][-MAX_SEQ_LEN:]
    pad_len = MAX_SEQ_LEN - len(seq)
    input_seq = torch.tensor([[PAD] * pad_len + seq], dtype=torch.long, device=device)

    sasrec_model.eval()
    with torch.no_grad():
        hidden = sasrec_model(input_seq)[0, -1]  # no nan_to_num needed here anymore -- forward() is clean now
        cand_tensor = torch.tensor([i + 1 for i in candidate_item_idx], dtype=torch.long, device=device)
        scores = (hidden.unsqueeze(0) * sasrec_model.item_emb(cand_tensor)).sum(-1)

    return scores.cpu().numpy()


def cf_sasrec_rerank(user_idx, N, candidate_pool_size=200):
    visitor_id = idx_to_user[user_idx]
    candidates = item_item_recommend(user_idx, candidate_pool_size, exclude_seen=False)

    scores = sasrec_score_candidates(visitor_id, candidates)
    if scores is None:
        return candidates[:N]

    order = np.argsort(-scores)
    return [candidates[i] for i in order][:N]


cf_sasrec_results, _ = evaluate(cf_sasrec_rerank)
print("Item-item CF + SASRec re-rank:", cf_sasrec_results)


Item-item CF + SASRec re-rank: {'Recall@5': 0.12801308539944903, 'Recall@10': 0.2652376033057851, 'Recall@20': 0.5103305785123967, 'Recall@50': 0.8069903581267218, 'Recall@100': 0.9163223140495868}


## Step 12 — Compare every model

In [23]:
comparison_df = pd.DataFrame({
    "Popularity": pop_results,
    "Transaction Popularity": txn_pop_results,
    "User-User CF": uu_results,
    "Item-Item CF": item_item_results,
    "Item-Item CF + SASRec": cf_sasrec_results,
    "ALS": als_results,
}).T[[f"Recall@{k}" for k in K_VALUES]]

print("PRIMARY METRIC -- Next-Interaction Recall@K (exclude_seen=False, full test set)")
comparison_df


PRIMARY METRIC -- Next-Interaction Recall@K (exclude_seen=False, full test set)


,Recall@5,Recall@10,Recall@20,Recall@50,Recall@100
Popularity,0.007145,0.010331,0.019972,0.037104,0.058798
Transaction Popularity,0.010933,0.017390,0.029528,0.048037,0.065857
User-User CF,0.000430,0.000430,0.000775,0.001119,0.001291
Item-Item CF,0.849862,0.890410,0.917786,0.941374,0.974518
Item-Item CF + SASRec,0.128013,0.265238,0.510331,0.806990,0.916322
ALS,0.235623,0.301136,0.388688,0.513085,0.604855


In [24]:
print(f"SECONDARY METRIC -- Discovery Recall@K (exclude_seen=True, n={n_novel_eval} novel-target users only)")
print("Not comparable to the table above -- different objective, different (much smaller) evaluation set.")
print(item_item_discovery_results)


SECONDARY METRIC -- Discovery Recall@K (exclude_seen=True, n=140 novel-target users only)
Not comparable to the table above -- different objective, different (much smaller) evaluation set.
{'Recall@5': 0.14285714285714285, 'Recall@10': 0.15, 'Recall@20': 0.22142857142857142, 'Recall@50': 0.24285714285714285, 'Recall@100': 0.2785714285714286}


**Read this before anything goes on a slide:** the primary table measures
next-interaction prediction on the full, honestly-evaluated test set. The
discovery number is a real but small-sample secondary metric -- label it
clearly as such if you use it, since it's evaluated on a different (much
smaller) subset of users than every other row.

If "Item-Item CF + SASRec" doesn't clearly beat "Item-Item CF" alone in the
primary table, that's a legitimate finding, not a failure -- report it
honestly rather than switching back to the broken `exclude_seen=True`
evaluation to make the SASRec row look better. That number would be
measuring a bug, not a model.


## Step 13 — Final production `recommend()`

In [25]:
def recommend(visitor_id, N=10, exclude_seen=False, use_sasrec=True):
    if visitor_id not in user_to_idx:
        return [
            {"item_id": int(i), "score": None, "source": "popularity_fallback"}
            for i in popular_item_ids[:N]
        ]

    user_idx = user_to_idx[visitor_id]

    if use_sasrec:
        candidates = item_item_recommend(user_idx, 200, exclude_seen=exclude_seen)
        scores = sasrec_score_candidates(visitor_id, candidates)
        if scores is not None:
            order = np.argsort(-scores)
            ranked = [candidates[i] for i in order][:N]
            return [
                {"item_id": int(idx_to_item[int(i)]), "score": float(scores[order[r]]), "source": "cf_sasrec_rerank"}
                for r, i in enumerate(ranked)
            ]

    ranked_idx, cf_scores = item_model.recommend(
        userid=user_idx, user_items=user_item_matrix[user_idx],
        N=N, filter_already_liked_items=exclude_seen,
    )
    return [
        {"item_id": int(idx_to_item[int(i)]), "score": float(s), "source": "item_item_cf"}
        for i, s in zip(ranked_idx, cf_scores)
    ]


results = recommend(172, N=10)
print("Visitor: 172\n")
print("Recommended Products:")
for rank, r in enumerate(results, 1):
    score_str = f"{r['score']:.4f}" if r['score'] is not None else "n/a"
    print(f"{rank}. Item {r['item_id']}   (score={score_str}, source={r['source']})")


Visitor: 172

Recommended Products:
1. Item 119736   (score=6.4077, source=cf_sasrec_rerank)
2. Item 9877   (score=6.3910, source=cf_sasrec_rerank)
3. Item 234255   (score=6.3026, source=cf_sasrec_rerank)
4. Item 320130   (score=6.2571, source=cf_sasrec_rerank)
5. Item 130815   (score=6.1636, source=cf_sasrec_rerank)
6. Item 384302   (score=6.0775, source=cf_sasrec_rerank)
7. Item 65273   (score=6.0492, source=cf_sasrec_rerank)
8. Item 461686   (score=5.9241, source=cf_sasrec_rerank)
9. Item 355994   (score=5.7354, source=cf_sasrec_rerank)
10. Item 439963   (score=5.6391, source=cf_sasrec_rerank)


## Step 14 — Save production artifacts

In [26]:
import pickle
from pathlib import Path
from scipy.sparse import save_npz

MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

with open(MODEL_DIR / 'item_model.pkl', 'wb') as f:
    pickle.dump(item_model, f)
save_npz(MODEL_DIR / 'user_item_matrix.npz', user_item_matrix)
with open(MODEL_DIR / 'mappings.pkl', 'wb') as f:
    pickle.dump({
        'user_to_idx': user_to_idx, 'item_to_idx': item_to_idx,
        'idx_to_user': idx_to_user, 'idx_to_item': idx_to_item,
    }, f)
with open(MODEL_DIR / 'popular_item_ids.pkl', 'wb') as f:
    pickle.dump(popular_item_ids, f)

torch.save(sasrec_model.state_dict(), MODEL_DIR / 'sasrec_model.pt')
with open(MODEL_DIR / 'sasrec_config.pkl', 'wb') as f:
    pickle.dump({'n_items': n_items, 'max_len': MAX_SEQ_LEN}, f)
with open(MODEL_DIR / 'last_session_by_user.pkl', 'wb') as f:
    pickle.dump(last_session_by_user, f)

print(f"Saved model artifacts to: {MODEL_DIR.resolve()}")
for p in sorted(MODEL_DIR.iterdir()):
    print(f"  - {p.name}  ({p.stat().st_size / 1024:.1f} KB)")


Saved model artifacts to: C:\Users\Ayush\Machine Learning projects\shope_sense\models
  - item_model.pkl  (40284.4 KB)
  - last_session_by_user.pkl  (17889.3 KB)
  - mappings.pkl  (54021.5 KB)
  - popular_item_ids.pkl  (1081.2 KB)
  - sasrec_config.pkl  (0.0 KB)
  - sasrec_model.pt  (59053.7 KB)
  - user_item_matrix.npz  (8029.9 KB)


## Summary

- **Objective:** next-interaction prediction, matching the actual train/test
  split -- not novel-item discovery.
- **`exclude_seen`:** `False` for the primary metric and production
  `recommend()`. `True` is available as a secondary, correctly-scoped
  metric (novel-target users only), not as the headline number.
- **Bugs fixed:** leakage, float64 dtype, session ordering, the
  `filter_already_liked_items` padding artifact, and the SASRec NaN leak
  between transformer layers.
- **Ship decision:** compare the "Item-Item CF" and "Item-Item CF + SASRec"
  rows in Step 12's real output on your data. Whichever genuinely wins is
  what `use_sasrec` should default to in Step 13 -- decided by the numbers,
  not assumed in advance.


In [27]:
def repeat_own_history_recommend(user_idx, N):
    """Trivial baseline: no model at all, just the user's own training
    items sorted by interaction weight. If this scores close to Item-Item
    CF, the CF row isn't demonstrating real collaborative signal -- it's
    mostly just successfully repeating the user's own history."""
    row = user_item_matrix[user_idx]
    order = np.argsort(-row.data)
    return list(row.indices[order][:N])

repeat_results, _ = evaluate(repeat_own_history_recommend)
print("Trivial repeat-own-history baseline:", repeat_results)

Trivial repeat-own-history baseline: {'Recall@5': 0.9616046831955923, 'Recall@10': 0.9752926997245179, 'Recall@20': 0.9803719008264463, 'Recall@50': 0.9841597796143251, 'Recall@100': 0.9858815426997245}
